# 3D-latent Euclidean VAE on Colab GPU

**Runtime -> Change runtime type -> T4 GPU** before running.

Two fixes vs the original script:

* `grid=100` removed from the `run_diagnostics_3d` calls -- that function has
  no `grid` parameter and the calls raise `TypeError`.
* `cyclic=(False, True)` for the sphere run: its polar angle is on [0, pi] and
  `hsv` would paint both poles the same, faking a seam.

Figures are saved exactly as named, one PDF per call.


In [1]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

Tue Aug 11 13:30:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Bachelorarbeit/projects'   # your .py files
OUT     = '/content/drive/MyDrive/vae_results/3d'
%cd $PROJECT
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Bachelorarbeit/projects
bin_scripts		    __pycache__		 topo_vae_run_analyse.ipynb
curvature_plot.py	    runstore.py		 trial
datasets.py		    sandbox_log.py	 vaediagnostic.py
flat_vae_run_analyse.ipynb  sandbox.py		 vaediagnostics_3d.py
new_sph_vae_chart.py	    standardvae_3dim.py  vis_datasets.py
new_tor_vae_charts.py	    standardvae.py


In [3]:
import os, sys, gc, time
sys.path.insert(0, os.getcwd())
os.makedirs(OUT, exist_ok=True)

import numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt

from vaediagnostics_3d import run_diagnostics_3d
from datasets import make_torus_dataset, make_plane_dataset, make_plane_dataset_old
try:
    from bin_scripts.sphere_data import make_sphere_dataset_try3
except ModuleNotFoundError as e:
    print(f"!! bin_scripts missing ({e}) -- falling back to a DIFFERENT dataset")
    from datasets import make_sphere_dataset_new as make_sphere_dataset_try3

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device {DEV}   out {OUT}")
if DEV != "cuda":
    print("!! no GPU attached")

device cuda   out /content/drive/MyDrive/vae_results/3d


In [4]:
class simpleVAE(nn.Module):
    def __init__(self, data_dim=2, latent_dim=3, hidden=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, data_dim),
        )

    def encode(self, data):
        h = self.encoder(data)          # one backbone pass, not two
        return self.mu(h), self.logvar(h)

    def reparametrize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, theta):
        return self.decoder(theta)

    def forward(self, data):
        mu, logvar = self.encode(data)
        return self.decode(self.reparametrize(mu, logvar)), mu, logvar


def elbo(x_recon, x, mu, logvar, beta=1.0):
    recon = ((x_recon - x) ** 2).sum(dim=1).mean()
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1).mean()
    return recon + beta * kl, recon.item(), kl.item()


def train(epochs=400, lr=1e-3, beta=1.0, dataset=None, device=DEV,
          seed=42, log_every=100):
    # seeded per run, so results do not depend on which cells ran before
    if seed is not None:
        torch.manual_seed(seed)
    imgs, angles = dataset()
    data = torch.as_tensor(imgs, dtype=torch.float32, device=device)
    if data.dim() > 2:
        data = data.flatten(1)

    model = simpleVAE(data_dim=data.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        opt.zero_grad(set_to_none=True)
        x_recon, mu, logvar = model(data)
        loss, r, k = elbo(x_recon, data, mu, logvar, beta)
        loss.backward(); opt.step()
        if log_every and (ep % log_every == 0 or ep == epochs - 1):
            print(f"epoch {ep:5d}  loss={loss.item():.4f}  recon={r:.4f}  kl={k:.4f}")
    return model, data, angles

In [5]:
def run(fname, dataset, epochs, cyclic, label):
    print(f"\n=== {fname}  ({label})")
    t0 = time.time()
    model, data, angles = train(epochs=epochs, dataset=dataset)
    print(f"    trained in {time.time()-t0:.0f}s")

    lat, sv = run_diagnostics_3d(model, data, angles, cyclic=cyclic,
                                 fname=os.path.join(OUT, fname))
    print(f"    wrote {fname}   total {time.time()-t0:.0f}s")
    plt.close('all')
    del model, data; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return lat, sv

## Run

One cell each. The spectrum panel is the point: if the third singular value
does not collapse, the codes genuinely occupy three dimensions.

In [6]:
_ = run("latent_3d_flat_with_borders_on_flat.pdf", make_plane_dataset_old,
        10000, (True, True), "flat plane, no border contact")


=== latent_3d_flat_with_borders_on_flat.pdf  (flat plane, no border contact)
epoch     0  loss=53.3770  recon=53.3574  kl=0.0196
epoch   100  loss=8.9789  recon=7.3716  kl=1.6073
epoch   200  loss=6.0535  recon=3.6926  kl=2.3609
epoch   300  loss=4.8495  recon=2.2249  kl=2.6246
epoch   400  loss=4.3541  recon=1.6066  kl=2.7475
epoch   500  loss=4.1734  recon=1.3724  kl=2.8010
epoch   600  loss=4.0944  recon=1.3017  kl=2.7926
epoch   700  loss=4.0466  recon=1.2460  kl=2.8006
epoch   800  loss=3.9974  recon=1.1782  kl=2.8192
epoch   900  loss=3.9942  recon=1.1683  kl=2.8258
epoch  1000  loss=3.9704  recon=1.1248  kl=2.8456
epoch  1100  loss=3.9299  recon=1.0940  kl=2.8358
epoch  1200  loss=3.9262  recon=1.0768  kl=2.8494
epoch  1300  loss=3.9479  recon=1.1134  kl=2.8346
epoch  1400  loss=3.9186  recon=1.0742  kl=2.8444
epoch  1500  loss=3.9286  recon=1.0847  kl=2.8439
epoch  1600  loss=3.8958  recon=1.0442  kl=2.8516
epoch  1700  loss=3.9231  recon=1.0656  kl=2.8575
epoch  1800  loss=3.

In [7]:
_ = run("latent_3d_flat_on_flat.pdf", make_plane_dataset,
        10000, (True, True), "flat plane, border contact")


=== latent_3d_flat_on_flat.pdf  (flat plane, border contact)
epoch     0  loss=49.6349  recon=49.6132  kl=0.0216
epoch   100  loss=19.5992  recon=17.9661  kl=1.6331
epoch   200  loss=16.3575  recon=13.4720  kl=2.8855
epoch   300  loss=12.7420  recon=8.6227  kl=4.1193
epoch   400  loss=10.6923  recon=6.0043  kl=4.6880
epoch   500  loss=9.7366  recon=4.8331  kl=4.9035
epoch   600  loss=9.1290  recon=4.1136  kl=5.0154
epoch   700  loss=8.6845  recon=3.6070  kl=5.0775
epoch   800  loss=8.3343  recon=3.2179  kl=5.1164
epoch   900  loss=8.1354  recon=2.9881  kl=5.1472
epoch  1000  loss=7.9639  recon=2.7720  kl=5.1919
epoch  1100  loss=7.7927  recon=2.5798  kl=5.2129
epoch  1200  loss=7.7104  recon=2.5023  kl=5.2080
epoch  1300  loss=7.5870  recon=2.3645  kl=5.2225
epoch  1400  loss=7.4715  recon=2.2185  kl=5.2531
epoch  1500  loss=7.4213  recon=2.1834  kl=5.2379
epoch  1600  loss=7.3287  recon=2.0696  kl=5.2591
epoch  1700  loss=7.2919  recon=2.0293  kl=5.2625
epoch  1800  loss=7.2105  reco

In [8]:
_ = run("latent_3d_torus_on_flat.pdf", make_torus_dataset,
        10000, (True, True), "torus")


=== latent_3d_torus_on_flat.pdf  (torus)
epoch     0  loss=53.1137  recon=53.0919  kl=0.0218
epoch   100  loss=21.6336  recon=19.8843  kl=1.7493
epoch   200  loss=17.7895  recon=14.5017  kl=3.2878
epoch   300  loss=13.9848  recon=9.6987  kl=4.2861
epoch   400  loss=11.9749  recon=7.1789  kl=4.7960
epoch   500  loss=10.8307  recon=5.8058  kl=5.0249
epoch   600  loss=9.9033  recon=4.7274  kl=5.1759
epoch   700  loss=9.2939  recon=4.0350  kl=5.2589
epoch   800  loss=8.8402  recon=3.5161  kl=5.3242
epoch   900  loss=8.5336  recon=3.1658  kl=5.3678
epoch  1000  loss=8.2845  recon=2.8742  kl=5.4102
epoch  1100  loss=8.0740  recon=2.6625  kl=5.4115
epoch  1200  loss=7.9592  recon=2.5358  kl=5.4233
epoch  1300  loss=7.8510  recon=2.4517  kl=5.3994
epoch  1400  loss=7.7491  recon=2.3335  kl=5.4156
epoch  1500  loss=7.6775  recon=2.2902  kl=5.3873
epoch  1600  loss=7.5828  recon=2.1561  kl=5.4268
epoch  1700  loss=7.5286  recon=2.1216  kl=5.4070
epoch  1800  loss=7.4445  recon=2.0332  kl=5.4114

In [9]:
_ = run("latent_3d_sphere_on_flat.pdf", make_sphere_dataset_try3,
        10000, (False, True), "sphere")


=== latent_3d_sphere_on_flat.pdf  (sphere)
epoch     0  loss=46.4940  recon=46.4730  kl=0.0210
epoch   100  loss=17.5022  recon=16.3243  kl=1.1779
epoch   200  loss=14.9949  recon=12.9435  kl=2.0514
epoch   300  loss=13.3356  recon=10.5883  kl=2.7473
epoch   400  loss=11.3991  recon=7.6331  kl=3.7660
epoch   500  loss=9.8203  recon=5.4522  kl=4.3681
epoch   600  loss=9.1206  recon=4.5432  kl=4.5774
epoch   700  loss=8.7768  recon=4.1102  kl=4.6666
epoch   800  loss=8.4396  recon=3.6924  kl=4.7472
epoch   900  loss=8.1318  recon=3.2854  kl=4.8464
epoch  1000  loss=7.8824  recon=2.9569  kl=4.9255
epoch  1100  loss=7.6666  recon=2.7091  kl=4.9575
epoch  1200  loss=7.5505  recon=2.5596  kl=4.9910
epoch  1300  loss=7.4587  recon=2.4672  kl=4.9915
epoch  1400  loss=7.3470  recon=2.3485  kl=4.9985
epoch  1500  loss=7.2987  recon=2.2814  kl=5.0172
epoch  1600  loss=7.1984  recon=2.1614  kl=5.0370
epoch  1700  loss=7.1556  recon=2.1186  kl=5.0370
epoch  1800  loss=7.1034  recon=2.0589  kl=5.04